## Descriptive Statistics

In [16]:
import pandas as pd

df = pd.read_csv("dataset/all_ai_models.csv")

print(f"Shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.describe(include='all')

Shape: (3237, 57)

Column names:
['Model', 'Domain', 'Task', 'Organization', 'Authors', 'Publication date', 'Reference', 'Link', 'Citations', 'Notability criteria', 'Notability criteria notes', 'Parameters', 'Parameters notes', 'Training compute (FLOP)', 'Training compute notes', 'Training dataset size (total)', 'Dataset size notes', 'Training time (hours)', 'Training time notes', 'Training hardware', 'Approach', 'Confidence', 'Abstract', 'Epochs', 'WikiText and Penn Treebank data', 'Model accessibility', 'Country (of organization)', 'Base model', 'Finetune compute (FLOP)', 'Finetune compute notes', 'Hardware quantity', 'Hardware utilization (MFU)', 'Last modified', 'Training cloud compute vendor', 'Training data center', 'Archived links', 'Batch size', 'Batch size notes', 'Organization categorization', 'Foundation model', 'Training compute lower bound', 'Training compute upper bound', 'Training chip-hours', 'Training code accessibility', 'Accessibility notes', 'Possibly over 1e23 FLOP

,Model,Domain,Task,Organization,Authors,Publication date,Reference,Link,Citations,Notability criteria,...,Utilization notes,Numerical format,Frontier model,Training power draw (W),Training compute estimation method,Hugging Face developer id,Post-training compute (FLOP),Post-training compute notes,Hardware utilization (HFU),Open model weights?
count,3237,3153,3119,3157,2466,3219,3076,3202,1270.000000,932,...,88,339,137,7.650000e+02,1438,609,1.000000e+00,1,25.000000,2488
unique,3236,172,894,1245,1919,1540,2453,2639,NaN,39,...,84,6,1,NaN,38,192,NaN,1,NaN,2
top,Tulu 3 (Tülu 3) 70B,Language,Language modeling,OpenAI,Qwen Team,2024-09-24,CAC registry,https://arxiv.org/abs/2205.01068,NaN,SOTA improvement,...,"Table 4: Training metric comparison\n\n""Causal...",BF16,True,NaN,Operation counting,Qwen,NaN,Section 4 gives detail about the post-training...,NaN,Yes
freq,2,1415,274,108,16,16,48,14,NaN,447,...,2,123,137,NaN,638,55,NaN,1,NaN,1260
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4147.406299,NaN,...,NaN,NaN,NaN,7.719242e+05,NaN,NaN,9.400000e+22,NaN,0.490876,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14666.204339,NaN,...,NaN,NaN,NaN,4.887626e+06,NaN,NaN,NaN,NaN,0.148841,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,...,NaN,NaN,NaN,7.582593e+01,NaN,NaN,9.400000e+22,NaN,0.235900,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.250000,NaN,...,NaN,NaN,NaN,4.153310e+03,NaN,NaN,9.400000e+22,NaN,0.400000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,409.000000,NaN,...,NaN,NaN,NaN,2.551799e+04,NaN,NaN,9.400000e+22,NaN,0.493500,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2441.750000,NaN,...,NaN,NaN,NaN,2.024958e+05,NaN,NaN,9.400000e+22,NaN,0.577200,NaN


## Cleaning

In [17]:
import pandas as pd

df = pd.read_csv("dataset/all_ai_models.csv")

results = []
for col in df.columns:
    s = df[col].astype(str).str.strip()
    has_true  = s.str.lower().eq("true").any()
    has_false = s.str.lower().eq("false").any()
    has_empty = (s.eq("") | df[col].isna()).any()
    if has_true or has_false:
        results.append({
            "column": col,
            "has_true":  has_true,
            "has_false": has_false,
            "has_empty": has_empty,
            "empty_means_false": has_true and not has_false and has_empty,
        })

pd.DataFrame(results).set_index("column")

,has_true,has_false,has_empty,empty_means_false
column,,,,
Foundation model,True,False,True,True
Possibly over 1e23 FLOP,True,False,True,True
Frontier model,True,False,True,True


**Encoding note — `True`/empty boolean columns:**
Three columns (`Foundation model`, `Frontier model`, `Possibly over 1e23 FLOP`) store booleans as `"True"` or blank — there are no `"False"` values at all. Empty therefore means False. These are recoded to `1`/`0` in the cleaned dataset.

In [18]:
true_empty_cols = ["Foundation model", "Frontier model", "Possibly over 1e23 FLOP"]

cleaned = df.copy()
for col in true_empty_cols:
    cleaned[col] = cleaned[col].astype(str).str.strip().eq("True").astype(int)

cleaned.to_csv("dataset/cleaned_ai_models.csv", index=False)
print(f"Saved {len(cleaned)} rows to dataset/cleaned_ai_models.csv")
cleaned[true_empty_cols].value_counts().head(10)

Saved 3237 rows to dataset/cleaned_ai_models.csv


Foundation model  Frontier model  Possibly over 1e23 FLOP
0                 0               0                          2614
                                  1                           424
                  1               0                            92
1                 0               1                            33
0                 1               1                            31
1                 0               0                            29
                  1               1                            13
                                  0                             1
Name: count, dtype: int64

## Correlation Matrix

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("dataset/cleaned_ai_models.csv")

# --- Encode variables ---
df["open"] = df["Open model weights?"].map({"Yes": 1, "No": 0})
df["log_compute"] = np.log10(pd.to_numeric(df["Training compute (FLOP)"], errors="coerce"))
df["log_params"]  = np.log10(pd.to_numeric(df["Parameters"], errors="coerce"))
df["log_cost"]    = np.log10(pd.to_numeric(df["Training compute cost (2023 USD)"], errors="coerce"))
df["year"]        = pd.to_datetime(df["Publication date"], errors="coerce").dt.year
df["is_industry"] = df["Organization categorization"].str.lower().str.contains("industry", na=False).astype(float)
df["citations"]   = pd.to_numeric(df["Citations"], errors="coerce")

cols = {
    "open":           "Open Weights",
    "log_compute":    "log(Compute)",
    "log_params":     "log(Parameters)",
    "log_cost":       "log(Cost USD)",
    "year":           "Year",
    "is_industry":    "Industry Org",
    "Frontier model": "Frontier Model",
    "citations":      "Citations",
}

corr_df = df[list(cols.keys())].rename(columns=cols)
corr = corr_df.corr()

print(f"N (rows with ≥1 value): {len(corr_df)}")
print(f"N (complete cases): {corr_df.dropna().shape[0]}\n")

plt.rcParams["font.family"] = "DejaVu Sans"

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
    linewidths=0.6, ax=ax, mask=mask, vmin=-1, vmax=1,
    annot_kws={"size": 26, "weight": "bold", "family": "DejaVu Sans"},
    cbar_kws={"shrink": 0.8},
)

ax.set_title("Pairwise Correlations (Pearson)", fontsize=30, fontweight="bold", pad=26)

ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=22, fontweight="bold")
ax.set_yticklabels(ax.get_yticklabels(), rotation=25, ha="right", fontsize=22, fontweight="bold")

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=20, width=1.5)
for label in cbar.ax.get_yticklabels():
    label.set_fontweight("bold")
    label.set_fontsize(20)

plt.tight_layout()
plt.show()